# 🧾 GLM-OCR → Excel Pipeline
**Image of a table → GLM-OCR → HTML → Formatted Excel (.xlsx)**

### Setup
Before running: `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────────────────
!pip install -q "transformers>=4.47.0" torch pillow openpyxl beautifulsoup4 markdown

In [ ]:
# ── CELL 1: Install dependencies (install transformers from source — required for GLM-OCR) ──
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q torch pillow openpyxl beautifulsoup4 markdown accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# ── CELL 2: Load GLM-OCR model ────────────────────────────────────────────
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_PATH = "zai-org/GLM-OCR"

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)

print("Loading model...")
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.eval()

print(f"\n✅ Model loaded on: {next(model.parameters()).device}")
print(f"   VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading processor...
Loading model...



✅ Model loaded on: cuda:0
   VRAM used: 4.66 GB


In [ ]:
import os
import gc
import tempfile
import torch
from PIL import Image, ImageOps

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

def extract_table_markdown(image_path: str) -> str:
    image = Image.open(image_path)
    if image.mode in ("RGBA", "LA", "P"):
        image = image.convert("RGB")
    image = ImageOps.exif_transpose(image)
    max_dim = 1920
    w, h = image.size
    if max(w, h) > max_dim:
        scale = max_dim / max(w, h)
        image = image.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".png")
    image.save(tmp.name, "PNG")
    tmp.close()
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "url": tmp.name},
            {"type": "text",  "text": "Table Recognition:"},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    inputs.pop("token_type_ids", None)
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=8192,
            use_cache=True,
        )
    output_text = processor.decode(
        generated_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    os.unlink(tmp.name)
    del inputs, generated_ids
    gc.collect()
    torch.cuda.empty_cache()
    return output_text.strip()
print("✅ GLM-OCR function ready")

✅ GLM-OCR function ready


In [ ]:
from bs4 import BeautifulSoup
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

HEADER_FILL    = PatternFill('solid', start_color='1F4E79', end_color='1F4E79')
SUBHEADER_FILL = PatternFill('solid', start_color='2E75B6', end_color='2E75B6')
ALT_ROW_FILL   = PatternFill('solid', start_color='DEEAF1', end_color='DEEAF1')
WHITE_FILL     = PatternFill('solid', start_color='FFFFFF', end_color='FFFFFF')
HEADER_FONT    = Font(name='Arial', bold=True, color='FFFFFF', size=10)
SUBHEADER_FONT = Font(name='Arial', bold=True, color='FFFFFF', size=9)
DATA_FONT      = Font(name='Arial', size=9)
TOTAL_FONT     = Font(name='Arial', bold=True, color='FFFFFF', size=9)
THIN_SIDE      = Side(style='thin', color='BFBFBF')
THIN_BORDER    = Border(left=THIN_SIDE, right=THIN_SIDE, top=THIN_SIDE, bottom=THIN_SIDE)
CENTER_ALIGN   = Alignment(horizontal='center', vertical='center', wrap_text=True)
LEFT_ALIGN     = Alignment(horizontal='left',   vertical='center', wrap_text=True)

def parse_html_table(html_source):
    soup  = BeautifulSoup(html_source, 'html.parser')
    table = soup.find('table')
    if not table:
        raise ValueError('No <table> found in HTML.')
    def parse_group(tag):
        rows = []
        for tr in tag.find_all('tr', recursive=False):
            cells = []
            for cell in tr.find_all(['th', 'td'], recursive=False):
                cells.append({
                    'text':    cell.get_text(strip=True),
                    'rowspan': int(cell.get('rowspan', 1)),
                    'colspan': int(cell.get('colspan', 1)),
                })
            rows.append(cells)
        return rows
    thead = table.find('thead')
    tbody = table.find('tbody')
    header_rows = parse_group(thead) if thead else []
    data_rows   = [[c['text'] for c in row] for row in parse_group(tbody)] if tbody else []
    return header_rows, data_rows

def html_to_excel(html_source, output_path='output.xlsx', sheet_name='Sheet1'):
    header_rows, data_rows = parse_html_table(html_source)
    wb = Workbook()
    ws = wb.active
    ws.title = sheet_name
    wb.save(output_path)
    return output_path

print("✅ Excel converter ready")

✅ Excel converter ready


In [ ]:
from google.colab import files
print("Upload your table image (JPG, PNG, WEBP)...")
# Note: This widget will not render on GitHub as it is a Colab-specific tool
try:
    uploaded   = files.upload()
    image_path = list(uploaded.keys())[0]
    print(f"\n✅ Uploaded: {image_path}")
except:
    print("Upload skipped.")

Upload your table image (JPG, PNG, WEBP)...


In [ ]:
import markdown as md_lib
from pathlib import Path
from IPython.display import display, HTML, Markdown
try:
    output_filename = Path(image_path).stem + '.xlsx'
    md_table = extract_table_markdown(image_path)
    html_table = md_lib.markdown(md_table, extensions=["tables"])
    html_to_excel(html_table, output_filename, sheet_name=Path(image_path).stem)
    print(f"\n✅ Done! File saved as: {output_filename}")
except NameError:
    print("No image uploaded yet.")

In [ ]:
from google.colab import files
try:
    files.download(output_filename)
except NameError:
    print("No output file to download.")